# 02_04_Profiling and Cleaning `punctuality_raw` Dataset 

In [1]:
from pathlib import Path

import pandas as pd

import infrabel_punctuality as ip

In [2]:
PATH_INTERMEDIATE = Path("../data/intermediate/")

PATH_RAW = Path("../data/raw/")

# Table of Contents

- [4. PUNCTUALITY DATASET](#4-punctuality-dataset)

- [4.1. Data Profiling](#41-data-profiling)
    - [4.1.1. Overview](#411-overview)
    - [4.1.2. Profiling `PTCAR_NO`, `PTCAR_LG_NM_NL`, `RELATION`, `RELATION_DIRECTION`, and `TRAIN_SERV`](#412-profiling-ptcar_no-ptcar_lg_nm_nl-relation-relation_direction-and-train_serv)
    - [4.1.3. Profiling `CIRC_TYP`, `THOP1_COD`, `OP1_COD`, `LINE_NO_DEP`, `LINE_NO_ARR`, and `TRAIN_NO`](#413-profiling-circ_typ-thop1_cod-op1_cod-line_no_dep-line_no_arr-and-train_no)

- [4.2. Data Quality & Cleaning](#42-data-quality--cleaning)
    - [4.2.1. Cleaning String Values](#421-cleaning-string-values)
    - [4.2.2. Converting Types](#422-converting-types)
    - [4.2.3. Dropping Irrelevant Columns](#423-dropping-irrelevant-columns)
    - [4.2.4. Handling Missing Values in Datetime Columns](#424-handling-missing-values-in-datetime-columns)
        - [4.2.4.1. Handling Missing Values in Arrival Columns](#4241-handling-missing-values-in-arrival-columns)
        - [4.2.4.2. Handling Missing Values in Departure Columns](#4242-handling-missing-values-in-departure-columns)
    - [4.2.5. Checking for Orphan Station Keys between `punctuality` and `stations_cleaned`](#425-checking-for-orphan-station-keys-between-punctuality-and-stations_cleaned)
        - [4.2.5.1. Cross-validating `punctuality` and `stations_cleaned` via `PTCAR_NO` to Detect Orphan Keys](#4251-cross-validating-punctuality-and-stations_cleaned-via-ptcar_no-to-detect-orphan-keys)
        - [4.2.5.2. Cross-validating `punctuality` with the Source Dataset via `PTCAR_NO`](#4252-cross-validating-punctuality-with-the-source-dataset-via-ptcar_no)
        - [4.2.5.3. Cross-validating on Station Names with the Source Data](#4253-cross-validating-on-station-names-with-the-source-data)
        - [4.2.5.4. In-depth Analysis of the Orphan `PTCAR_NO` Keys](#4254-in-depth-analysis-of-the-orphan-ptcar_no-keys)

- [4.3. Export to Silver Layer](#43-export-to-silver-layer)

## 4. PUNCTUALITY DATASET

## 4.1. Data Profiling

### 4.1.1. Overview

This dataset is the future **Fact Table** of our star schema data warehouse.

As shown below:
* `punctuality` contains **45,542,084 entries** and **22 columns**.  

* It contains **no duplicates**.  

This large number of entries and the absence of duplicates are expected for a Fact Table. 

Among the columns of this dataset:
* `RELATION` and `RELATION_DIRECTION` are selected to build the future `train_service` dimension.  

* `PTCAR_NO` will be used as a **Foreign Key** with the `station` dimension.  

* The **various DATE and TIME columns** will point to the `date` and `time` dimensions, and will be essential to calculate punctuality metrics.

However, there is a large number of **missing values** in several columns. In the table below, we present the most significant ones for our analysis: 

| Columns                                                                      | Missing values      |
|------------------------------------------------------------------------------|---------------------|
| DATDEP, TRAIN_NO, RELATION, TRAIN_SERV, PTCAR_NO, PTCAR_NM_NL, CIRC_TYP      | No missing values   |
| RELATION_DIRECTION                                                           | 2,686,482           |
| REAL_TIME_ARR, PLANNED_TIME_ARR, PLANNED_DATE_ARR, REAL_DATE_ARR             | 2,254,669           |
| PLANNED_TIME_DEP, PLANNED_DATE_DEP                                           | 2,251,536           |
| REAL_TIME_DEP, REAL_DATE_DEP                                                 | 2,251,535           |

* Notably, `REAL_TIME_DEP` and `REAL_DATE_DEP` have one fewer missing values than their equivalent PLANNED columns - `PLANNED_TIME_DEP` and `PLANNED_DATE_DEP`. This discrepancy is handled in the 4.2.4. *Handling Missing Values in Date and Time Related Columns* section.  

* The remaining columns also contain missing values and are discussed in the profiling sections below.

In [3]:
punctuality = pd.read_parquet(PATH_INTERMEDIATE / "punctuality_raw.parquet")

In [4]:
punctuality.head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,THOP1_COD,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,...,DELAY_DEP,CIRC_TYP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,OP1_COD
0,01JAN2024,1813,IC 02,SNCB/NMBS,929,NaN,50A,NaN,13:09:42,NaN,...,42.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,NaN,NaN,01JAN2024,NaN,01JAN2024,NaN
1,01JAN2024,1813,IC 02,SNCB/NMBS,210,=,50A,13:22:17,13:25:20,13:22:00,...,20.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
2,01JAN2024,1813,IC 02,SNCB/NMBS,931,D,50A,13:29:18,13:29:18,13:28:00,...,78.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
3,01JAN2024,1813,IC 02,SNCB/NMBS,127,P,50A,13:31:59,13:31:59,13:31:00,...,59.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
4,01JAN2024,1813,IC 02,SNCB/NMBS,797,D,50A,13:33:57,13:33:57,13:33:00,...,57.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN


In [5]:
punctuality.info()

<class 'pandas.DataFrame'>
RangeIndex: 45542085 entries, 0 to 45542084
Data columns (total 22 columns):
 #   Column              Dtype  
---  ------              -----  
 0   DATDEP              str    
 1   TRAIN_NO            int64  
 2   RELATION            str    
 3   TRAIN_SERV          str    
 4   PTCAR_NO            int64  
 5   THOP1_COD           str    
 6   LINE_NO_DEP         str    
 7   REAL_TIME_ARR       str    
 8   REAL_TIME_DEP       str    
 9   PLANNED_TIME_ARR    str    
 10  PLANNED_TIME_DEP    str    
 11  DELAY_ARR           float64
 12  DELAY_DEP           float64
 13  CIRC_TYP            int64  
 14  RELATION_DIRECTION  str    
 15  PTCAR_LG_NM_NL      str    
 16  LINE_NO_ARR         str    
 17  PLANNED_DATE_ARR    str    
 18  PLANNED_DATE_DEP    str    
 19  REAL_DATE_ARR       str    
 20  REAL_DATE_DEP       str    
 21  OP1_COD             str    
dtypes: float64(2), int64(3), str(17)
memory usage: 13.2 GB


In [6]:
punctuality.isnull().sum()

DATDEP                       0
TRAIN_NO                     0
RELATION                     0
TRAIN_SERV                   0
PTCAR_NO                     0
THOP1_COD              4557929
LINE_NO_DEP            2251848
REAL_TIME_ARR          2254669
REAL_TIME_DEP          2251535
PLANNED_TIME_ARR       2254669
PLANNED_TIME_DEP       2251536
DELAY_ARR              2254574
DELAY_DEP              2251430
CIRC_TYP                     0
RELATION_DIRECTION     2686482
PTCAR_LG_NM_NL               0
LINE_NO_ARR            2254978
PLANNED_DATE_ARR       2254669
PLANNED_DATE_DEP       2251536
REAL_DATE_ARR          2254669
REAL_DATE_DEP          2251535
OP1_COD               38410974
dtype: int64

In [7]:
punctuality.nunique()

DATDEP                  731
TRAIN_NO               6614
RELATION                171
TRAIN_SERV                4
PTCAR_NO                692
THOP1_COD                10
LINE_NO_DEP             198
REAL_TIME_ARR         81450
REAL_TIME_DEP         81353
PLANNED_TIME_ARR      47417
PLANNED_TIME_DEP      43884
DELAY_ARR             12876
DELAY_DEP             12719
CIRC_TYP                  1
RELATION_DIRECTION      498
PTCAR_LG_NM_NL          692
LINE_NO_ARR             201
PLANNED_DATE_ARR        732
PLANNED_DATE_DEP        732
REAL_DATE_ARR           732
REAL_DATE_DEP           732
OP1_COD                  10
dtype: int64

**⚠️ Warning**: the cell below takes approximately 4 to 6 minutes to execute on a standard machine due to the size of the `punctuality` dataset.

In [8]:
punctuality.duplicated().sum()

np.int64(0)

### 4.1.2. Profiling `PTCAR_NO`, `PTCAR_LG_NM_NL`, `RELATION`, `RELATION_DIRECTION`, and `TRAIN_SERV`

As shown in the *Overview* section:  

* `RELATION` and `RELATION_DIRECTION` will be used to build the future `train_service` dimension.  
* `PTCAR_NO` will be used as a Foreign Key with the `station` dimension, more precisely with its `ptcarid` column.  
* `PTCAR_NO`, `RELATION`, `TRAIN_SERV`, and `PTCAR_LG_NM_NL` have no missing values.
* `RELATION DIRECTION` contains **2,686,482** missing values.  
* `RELATION` and `RELATION_DIRECTION` contain respectively **171** and **498** unique values.    

As the following section reveals:
* `PTCAR_NO` and `PTCAR_LG_NM_NL` both contain **692 unique values in the same rows**.  

* `TRAIN_SERV` contains **4** unique values related to train national or international operators:  
    - SNCB for Belgian trains  
    - EUROSTARFR for Eurostar in Belgium and France  
    - EURSLEEPER for European international night trains  
    - THI-FACT for THI Factory SA, which operates high-speed trains such as Thalys in the Belgian network  

* From `RELATION`, we can infer **11 relation categories**, 4 national and 7 international.

According to the SNCB documentation, these relation categories correspond to the following train types:  

| Relation category | SNCB Documentation                                                          |
|-------------------|-----------------------------------------------------------------------------|
|	IC              | InterCity trains in Belgium                                                 |
|	L               | Local trains, usually renamed "S" in public communication since 2015-2018   |
|	P               | Additional trains during peak hours or from university cities on weekend    |
|	EXTRA           | Additional trains in case of incident or significant delays on the network  |
|	ICE             | InterCityExpress from German cities, operated by Deutsche Bahn              |
|	INT             | Other international trains                                                  |
|	THAL            | Thalys trains between Brussels and Paris                                    |
|	EURST           | Eurostar trains between London and Brussels, Paris or Amsterdam             |
|	TGV             | French High-Speed Train (Train à Grande Vitesse)                            |
|	CHARTER         | International trains for specific events (e.g. Tomorrowland festival, Venice Simpleton Orient Express)                                                                                             |

<br>

**Decision**
* We keep `PTCAR_LG_NM_NL` temporary, in case is is needed to join `punctuality` and `station_cleaned` in the *build dimension station* notebook. Since this column matches perfectly `PTCAR_NO` and station names are already normalized in the `station` dimension, it will be excluded from the Fact Table after the dimension station is built.

* We keep `TRAIN_SERV` for the future `train_service` dimension, as it provides useful information about service operators.    

* When we build the `train_service` dimension in the *build secondary dimensions* notebook, a new binary column `Is_national` will be created based on the relation catagories, in order to refine our punctuality analysis.

* We investigate missing values in `RELATION DIRECTION` in sub-section 4.2.3.1. further in this notebook.  

* `punctuality` and `station_cleaned` are cross-validated on `PTCAR_NO` in the *02_05_handling_missing_values_relation_direction_punctuality* notebook.


In [9]:
ptcarid_unique = punctuality["PTCAR_NO"].nunique()
ptcar_nm_unique = punctuality["PTCAR_LG_NM_NL"].nunique()
relation_unique = punctuality["RELATION"].nunique()
relation_direction_unique = punctuality["RELATION_DIRECTION"].nunique()

print(f"PTCAR_LG_NM_NL contains {ptcar_nm_unique} unique values.")
print(f"PTCAR_NO contains {ptcarid_unique} unique values.")
print(f"RELATION and RELATION_DIRECTION contain respectively {relation_unique}"
      f" and {relation_direction_unique} unique values.")

PTCAR_LG_NM_NL contains 692 unique values.
PTCAR_NO contains 692 unique values.
RELATION and RELATION_DIRECTION contain respectively 171 and 498 unique values.


In [10]:
ptcar_unique = punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]].drop_duplicates().reset_index(drop=True)
ptcar_unique.shape

(692, 2)

The 692 unique values in `PTCAR_LG_NM_NL` and `PTCAR_NO` are in the same rows.

In [11]:
punctuality["RELATION_DIRECTION"].drop_duplicates().dropna().reset_index(drop=True)

0               IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL
1      IC 23-2: KNOKKE -> BRUSSELS AIRPORT - ZAVENTEM
2            IC 04-1: POPERINGE -> ANTWERPEN-CENTRAAL
3                         IC 03: GENK -> BLANKENBERGE
4                                L 16: NAMUR -> CINEY
                            ...                      
493                        S53: OUDENAARDE -> LOKEREN
494          S35: NOORDERKEMPEN -> ANTWERPEN-CENTRAAL
495                        S53: LOKEREN -> OUDENAARDE
496          S35: ANTWERPEN-CENTRAAL -> NOORDERKEMPEN
497                     S4: AALST -> LOUVAIN-LA-NEUVE
Name: RELATION_DIRECTION, Length: 498, dtype: str

In [12]:
services = punctuality["RELATION"].drop_duplicates().dropna().reset_index(drop=True)
services.head()

0      IC 02
1    IC 23-2
2    IC 04-1
3      IC 03
4       L 16
Name: RELATION, dtype: str

In [13]:
train_services = punctuality["TRAIN_SERV"].drop_duplicates().reset_index(drop=True)
train_services

0     SNCB/NMBS
1    EURSLEEPER
2      THI-FACT
3    EUROSTARFR
Name: TRAIN_SERV, dtype: str

In [14]:
services_categories = services.str.extract(r'^([A-Z]+)', expand=False)
services_categories = services_categories.drop_duplicates().reset_index(drop=True)
print(f"There are {services_categories.nunique()} service categories.")
services_categories

There are 11 service categories.


0          IC
1           L
2         ICE
3         INT
4        THAL
5       EURST
6         TGV
7       EXTRA
8           P
9     CHARTER
10          S
Name: RELATION, dtype: str

### 4.1.3. Profiling `CIRC_TYP`, `THOP1_COD`, `OP1_COD`, `LINE_NO_DEP`, `LINE_NO_ARR`, and `TRAIN_NO`

As the Overview section and the outputs below indicate:

* `CIRC_TYP` contains **a single unique integer value** `1` across all 45,542,084 records. As a zero-variance column, it carries no discriminating information and cannot contribute to any analysis.

* `THOP1_COD` and `OP1_COD` contain **10 identical unique values** consisting of single characters and operators ('P', '=', 'D', 'V', '(', ')', 'C', '<', '>', ':'), with respectively **~10% and ~85% missing values**.  
These columns appear to be internal operational codes from OP1 (Operation Platform 1), a third-party rail traffic management platform. As access to the OP1 platform is required to interpret these codes, their analytical meaning cannot be determined.  
Furthermore, with 85% missing values in OP1_COD, this column carries no meaningful information for our analysis.

* `LINE_NO_ARR` and `LINE_NO_DEP` contain the **infrastructure line codes** (e.g. 50A for "BRUXELLES-MIDI - OOSTEND") used by each train movement, as opposed to the train service identifiers in RELATION and RELATION_DIRECTION, which represent the commercial routes as perceived by passengers (e.g. "IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL").  
These columns are retained in the data warehouse for potential future analysis (e.g. punctuality by infrastructure line rather than by station or service), but are not used in the current analysis, which focuses on passenger-facing metrics, except for the in-depth analysis about `PTCAR_NO` orphan values between `punctuality` and `stations_cleaned` in the 4.2.5.4. sub-section at the end of this notebook.   
Their missing values (~5% for both columns) are not investigated at this stage, as no immediate analytical use case justifies the effort. This can be revisited if a future analysis requires it.  

* `TRAIN_NO` contains no missing values and 6614 train numbers. 
  * Train numbers are dynamic operational identifiers assigned to train path requests by Infrabel, not permanent identifiers of specific physical train sets. As documented in the Infrabel Network Reference Document (*Document de référence du réseau - Version du 02/04/2026*), train numbers can be reassigned during operations — for example, in *Appendix D05-RIEI-11*, among others, a freight train from Germany delayed by more than 10 hours is assigned a new train number in the 98500–98599 series. This confirms that train numbers are not stable unique identifiers across the dataset's time range. The **physical rolling stock** assigned to a train number may change during its journey due to coupling or decoupling of units, or if the train is rerouted following a network incident.    

  * However, the train numbers **assigned to a train service** are relatively stable for extended periods of time. Therefore, we will use it to handle missing values in `RELATION_DIRECTION`.
<br><br>

**Decisions**
* Therefore, `CIRC_TYP`, `THOP1_COD`, and `OP1_COD` are dropped, while `LINE_NO_ARR` and `LINE_NO_DEP` are kept without modification for potential future analysis.  

* `TRAIN_NO` is temporarily retained to handle missing values in `RELATION_DIRECTION` and will be dropped once these missing relation directions are handled.


In [15]:
punctuality["CIRC_TYP"].drop_duplicates().reset_index(drop=True)

0    1
Name: CIRC_TYP, dtype: int64

In [16]:
punctuality["THOP1_COD"].drop_duplicates().dropna().reset_index(drop=True).to_list()

['=', 'D', 'P', 'V', 'C', ')', '(', '>', '<', ':']

In [17]:
punctuality["OP1_COD"].drop_duplicates().dropna().reset_index(drop=True).to_list()

['P', '=', 'D', 'V', '(', ')', 'C', '<', '>', ':']

In [18]:
line_dep = punctuality["LINE_NO_DEP"].drop_duplicates().dropna().reset_index(drop=True)
line_dep.head()

0     50A
1     50E
2    58/1
3      58
4      59
Name: LINE_NO_DEP, dtype: str

In [19]:
line_arr = punctuality["LINE_NO_ARR"].drop_duplicates().dropna().reset_index(drop=True)
line_arr.head()

0      50A
1    50A/6
2     58/1
3       58
4       59
Name: LINE_NO_ARR, dtype: str

In [20]:
trains_by_relation = (
    punctuality.groupby("RELATION")["TRAIN_NO"].agg(total_relations="count", unique_trains="nunique")
)
trains_by_relation.sort_values("unique_trains", ascending=False)

,total_relations,unique_trains
RELATION,,
EXTRA,795494,1825
P,1770742,410
EURST,353556,188
THAL,258694,124
IC 23-2,655346,83
...,...,...
L 37-2,15380,16
L C2-2,34746,16
IC 41,785,6


## 4.2. Data Quality & Cleaning

To optimize memory layout, avoid an inconveniently extended notebook, and enable time-based exploration for the extensive proportion of missing values in the `RELATION_DIRECTION` column: 

* We perform an **initial type conversion on dates and specific keys**, deliberately keeping time columns as strings, as they will be converted during the fact table building phase, as explained in the section below.

* We drop irrelevant columns **early in the workflow**.

* We handle the missing values in `RELATION_DIRECTION` in **a separate notebook** -  *02_05_handling_missing_values_relation_direction_punctuality* - and focus on the missing values in the other `punctuality` columns in this one.

* We clean string columns **at bare minimum** in the 4.2.3. section for three reasons:  

    * The string complexities in `PTCAR_LG_NM_NL` is specifically addressed in the *4.2.5. Checking Orphans between `punctuality` and `stations_cleaned` on `PTCAR_NO` and `PTCAR_LG_NM_NL`* section.  
    
    * The complexity in the `RELATION_DIRECTION` string values will be handled in the *02_05_handling_missing_values_relation_direction_punctuality* notebook. 

    * Since many Infrabel referentials, such as the train service names in `RELATION`, the railway line IDs in `LINE_NO_ARR` and `LINE_NO_DEP`, the name operators in `TRAIN_SERV`, use special characters ("/", "-") or capital letters, we cannot strip them from the string columns without losing indispensable informations. Therefore, we only strip leading, trailing and double whitespaces in these columns.


### 4.2.1. Cleaning String Values

In [21]:
columns_to_strip = ["RELATION", "RELATION_DIRECTION", "TRAIN_SERV", "LINE_NO_DEP", "LINE_NO_ARR", "PTCAR_LG_NM_NL", 
                    "REAL_TIME_ARR", "PLANNED_TIME_ARR", "REAL_TIME_DEP", "PLANNED_TIME_DEP",
                    "REAL_DATE_ARR", "PLANNED_DATE_ARR", "REAL_DATE_DEP", "PLANNED_DATE_DEP"]

**⚠️ Warning**: the cell below may take approximately 2 to 4 minutes to execute on a standard machine due to the size of the `punctuality` dataset.

In [22]:
ip.strip_df_string(df=punctuality, columns=columns_to_strip).head()

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,THOP1_COD,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,...,DELAY_DEP,CIRC_TYP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,OP1_COD
0,01JAN2024,1813,IC 02,SNCB/NMBS,929,NaN,50A,NaN,13:09:42,NaN,...,42.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,NaN,NaN,01JAN2024,NaN,01JAN2024,NaN
1,01JAN2024,1813,IC 02,SNCB/NMBS,210,=,50A,13:22:17,13:25:20,13:22:00,...,20.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
2,01JAN2024,1813,IC 02,SNCB/NMBS,931,D,50A,13:29:18,13:29:18,13:28:00,...,78.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
3,01JAN2024,1813,IC 02,SNCB/NMBS,127,P,50A,13:31:59,13:31:59,13:31:00,...,59.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN
4,01JAN2024,1813,IC 02,SNCB/NMBS,797,D,50A,13:33:57,13:33:57,13:33:00,...,57.0,1,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,01JAN2024,01JAN2024,01JAN2024,01JAN2024,NaN


### 4.2.2. Converting Types

* `PTCAR_NO` is converted from `str` to `Int64` for future joins with the `stations_cleaned` dataset.  

* `DATDEP`, `PLANNED_DATE_ARR`, `PLANNED_DATE_DEP`, `REAL_DATE_ARR`, and `REAL_DATE_DEP` are converted from `str` to `datetime64`, as they are required as date objects for downsteam missing value handling.  

* `PLANNED_TIME_ARR`, `PLANNED_TIME_DEP`, `REAL_TIME_ARR`, and `REAL_TIME_DEP` are kept as `str` columns at this stage, as we need them to generate new derived `timedelta64` columns for delay metrics, before converting them to `Int64` **Surrogate Keys** linking the fact table to the time dimension. Both operations will be handled in the *building_fact_punctuality* notebook.

* Remaining `str` columns are converted to `string`, and the `int64` `TRAIN_NO` column into `Int64`, ensuring that pandas `NaN` values are properly mapped to `NULL` in SQL Server.

In [23]:
punctuality.dtypes

DATDEP                    str
TRAIN_NO                int64
RELATION                  str
TRAIN_SERV                str
PTCAR_NO                int64
THOP1_COD                 str
LINE_NO_DEP               str
REAL_TIME_ARR             str
REAL_TIME_DEP             str
PLANNED_TIME_ARR          str
PLANNED_TIME_DEP          str
DELAY_ARR             float64
DELAY_DEP             float64
CIRC_TYP                int64
RELATION_DIRECTION        str
PTCAR_LG_NM_NL            str
LINE_NO_ARR               str
PLANNED_DATE_ARR          str
PLANNED_DATE_DEP          str
REAL_DATE_ARR             str
REAL_DATE_DEP             str
OP1_COD                   str
dtype: object

In [24]:
punctuality["PTCAR_NO"] = punctuality["PTCAR_NO"].astype("Int64")

In [25]:
punctuality["TRAIN_NO"] = punctuality["TRAIN_NO"].astype("Int64")

In [26]:
punctuality[["DATDEP", "PLANNED_DATE_ARR", "PLANNED_DATE_DEP", "REAL_DATE_ARR", "REAL_DATE_DEP"]].tail( )

,DATDEP,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP
45542080,31DEC2025,31DEC2025,31DEC2025,31DEC2025,31DEC2025
45542081,31DEC2025,31DEC2025,NaN,31DEC2025,NaN
45542082,31DEC2025,NaN,31DEC2025,NaN,31DEC2025
45542083,31DEC2025,31DEC2025,31DEC2025,31DEC2025,31DEC2025
45542084,31DEC2025,31DEC2025,NaN,31DEC2025,NaN


In [27]:
columns_to_convert_to_date = ["DATDEP", "PLANNED_DATE_ARR", "PLANNED_DATE_DEP", "REAL_DATE_ARR", "REAL_DATE_DEP"]

In [28]:
punctuality[columns_to_convert_to_date] = (
    punctuality[columns_to_convert_to_date].apply(
        lambda x: pd.to_datetime(x.str.title(), format="%d%b%Y")
                    )
                )

In [29]:
punctuality[columns_to_convert_to_date].tail()

,DATDEP,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP
45542080,2025-12-31,2025-12-31,2025-12-31,2025-12-31,2025-12-31
45542081,2025-12-31,2025-12-31,NaT,2025-12-31,NaT
45542082,2025-12-31,NaT,2025-12-31,NaT,2025-12-31
45542083,2025-12-31,2025-12-31,2025-12-31,2025-12-31,2025-12-31
45542084,2025-12-31,2025-12-31,NaT,2025-12-31,NaT


In [30]:
string_columns = ["TRAIN_SERV", "RELATION", "RELATION_DIRECTION", "PTCAR_LG_NM_NL", 
                  "REAL_TIME_ARR", "PLANNED_TIME_ARR", "REAL_TIME_DEP", "PLANNED_TIME_DEP",
                  "LINE_NO_DEP", "LINE_NO_ARR"]

for col in string_columns:
    punctuality[col] = punctuality[col].astype("string")

In [31]:
punctuality.dtypes


DATDEP                datetime64[us]
TRAIN_NO                       Int64
RELATION                      string
TRAIN_SERV                    string
PTCAR_NO                       Int64
THOP1_COD                        str
LINE_NO_DEP                   string
REAL_TIME_ARR                 string
REAL_TIME_DEP                 string
PLANNED_TIME_ARR              string
PLANNED_TIME_DEP              string
DELAY_ARR                    float64
DELAY_DEP                    float64
CIRC_TYP                       int64
RELATION_DIRECTION            string
PTCAR_LG_NM_NL                string
LINE_NO_ARR                   string
PLANNED_DATE_ARR      datetime64[us]
PLANNED_DATE_DEP      datetime64[us]
REAL_DATE_ARR         datetime64[us]
REAL_DATE_DEP         datetime64[us]
OP1_COD                          str
dtype: object

### 4.2.3. Dropping Irrelevant Columns

As established in the 4.1.3. section, we drop the `THOP1_COD`, `OP1_COD`, and `CIRC_TYP` columns.

In [32]:
columns_to_drop = ["THOP1_COD", "OP1_COD", "CIRC_TYP"]

punctuality = punctuality.drop(columns=columns_to_drop)

### 4.2.4. Handling Missing Values in Datetime Columns

As noticed in the *Overview* section, there is a high volume of missing values in this dataset.

Consequently, we handle the missing values in `RELATION` and `RELATION_DIRECTION` in the next notebook *02_05_handling_missing_values_relation_direction_punctuality*, and focus this section on the missing values in the following ten date and time related columns: 
* `REAL_TIME_ARR`, `PLANNED_TIME_ARR`, `REAL_DATE_ARR`, `PLANNED_DATE_ARR`, `DELAY_ARR`
* `REAL_TIME_DEP`, `PLANNED_TIME_DEP`, `REAL_DATE_DEP`, `PLANNED_DATE_DEP`, `DELAY_DEP`

As shown in the two sub-sections below:  

* The missing values in `REAL_TIME_ARR`, `PLANNED_TIME_ARR`, `REAL_DATE_ARR`, and `PLANNED_DATE_ARR` are all in the same rows.  

* Except for one unique record (index `19843209`), the missing values in `REAL_TIME_DEP`, `PLANNED_TIME_DEP`, `REAL_DATE_DEP`, and `PLANNED_DATE_DEP` are all in the same rows. 

* The record at index `19843209` presents missing values in `PLANNED_TIME_DEP` and `PLANNED_DATE_DEP` while displaying values for `REAL_TIME_DEP` and `REAL_DATE_DEP`. This corresponds to the anomalous record we noted in the *Overview* section. As established above, this record presents a logical inconsistency: there is no value in `PLANNED_TIME_DEP`, yet `DELAY_DEP` shows a value of **71.0 minutes** - which is **structurally impossible** if `DELAY_DEP` were derived from the difference between real and planned departure times. Such a pattern suggests a data recording anomaly, possibly related to a day-boundary crossing on 2024-11-10 at LEUVEN. 

* Missing values in the _ARR and _DEP column groups are not errors but **not applicable (N/A) values**: they correspond to **trains at the end or beginning of their line**, for which no subsequent departure or previous arrival exists by definition. These values are therefore retained as `NaT` or `NaN` rather than being imputed or removed. As established in the project *README*, even though the current analysis focus on **arrival delays** as they are the most directly perceived by passengers, two SQL views will be created in the data warehouse:  

    * vw_punctuality_arrivals — filters out records where arrival data is unavailable, enabling arrival delay analysis.  

    * vw_punctuality_departures — filters out records where departure data is unavailable, enabling departure delay analysis.  

* The `DELAY_ARR` and `DELAY_DEP` columns reveal logical inconsistencies: for 200 rows, `0.0` values when there are `NaN` missing values in `REAL_TIME` and `PLANNED_TIME` columns. For the entry at index `19843209`, a value of `-71.0` is present despite a `NaN` missing value in `PLANNED_TIME`. These inconsistencies prove that, at least for these rows, `DELAY_ARR` and `DELAY_DEP` are not derivate columns calculated from the difference between `REAL_TIME` and `PLANNED_TIME` columns, but rather data imported for an external source.  


**Decisions**
* The logical inconsistencies in `DELAY_ARR` and `DELAY_DEP` columns confirm our decision to calculate our own derived delay columns, as we established in the project *README* of this project. This will be handled in **SQL** once the pandas DataFrames are loaded into the SQL data warehouse.  

* Altough the Infrabel `DELAY_ARR` and `DELAY_DEP` columns are not used in our analysis, as delay metrics will be recalculated form the raw arrival and departure time columns to ensure full control over the computation, we replace the inconsistent values with `NaN` for consistency.

* Given that the anomaly in the `19843209` row represents **a single record out of 45,542,084**, we **drop it without imputation**, as it will have no significant impact on statistical calculation (~0.000002% of the dataset records).

* Since they represent not applicable values, the missing values in **_ARR** and **_DEP** columns are retained as `NaT` or `NaN`. They will be converted to `NULL` in SQL data warehouse during the loading of the fact table in the *loading_fact_punctuality_to_sql_server* notebook. 

### 4.2.4.1. Handling Missing Values in Arrival Columns

In [33]:
# Checking NaN for _ARR columns
print(
    (
        (punctuality["REAL_TIME_ARR"].isnull() == punctuality["PLANNED_TIME_ARR"].isnull()) &
        (punctuality["PLANNED_TIME_ARR"].isnull() == punctuality["REAL_DATE_ARR"].isnull()) &
        (punctuality["REAL_DATE_ARR"].isnull() == punctuality["PLANNED_DATE_ARR"].isnull()) &
        (punctuality["PLANNED_DATE_ARR"].isnull() == punctuality["DELAY_ARR"].isnull())
    ).all()
)

False


In [34]:
# Finding the problematical _ARR columns
ref_nan_arr = punctuality["REAL_TIME_ARR"].isnull()

cols_arr = ["PLANNED_TIME_ARR", "REAL_DATE_ARR", "PLANNED_DATE_ARR", "DELAY_ARR"]

for col in cols_arr:
    match = (punctuality[col].isnull() == ref_nan_arr).all()
    print(f"{col} : {match}")

PLANNED_TIME_ARR : True
REAL_DATE_ARR : True
PLANNED_DATE_ARR : True
DELAY_ARR : False


In [35]:
#Finding the problematical rows in DELAY_ARR
arr_rows_diff = punctuality["REAL_TIME_ARR"].isnull() != punctuality["DELAY_ARR"].isnull()
print(f"Inconsistent rows: {arr_rows_diff.sum()}")

arr_columns_to_check = (
 ["DATDEP", "RELATION", "PTCAR_LG_NM_NL", "REAL_TIME_ARR", "PLANNED_TIME_ARR", 
  "REAL_DATE_ARR", "PLANNED_DATE_ARR", "DELAY_ARR"]
)

punctuality.loc[arr_rows_diff, arr_columns_to_check].head()

Inconsistent rows: 95


,DATDEP,RELATION,PTCAR_LG_NM_NL,REAL_TIME_ARR,PLANNED_TIME_ARR,REAL_DATE_ARR,PLANNED_DATE_ARR,DELAY_ARR
674694,2024-01-11,ICE,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0
732985,2024-01-12,ICE,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0
1154595,2024-01-19,ICE,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0
2022803,2024-02-01,L L2,LIERS,<NA>,<NA>,NaT,NaT,0.0
3202733,2024-02-20,ICE,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0


As shown above, the problematical rows in `DELAY_ARR` reveal a logical inconsistency: they display `0.0` values when `<Na>` is present in both `REAL_TIME_ARR` and `PLANNED_TIME_ARR`. 

In [36]:
punctuality.loc[arr_rows_diff, "DELAY_ARR"] = pd.NA

In [37]:
#Verification
arr_rows_diff_after = punctuality["REAL_TIME_ARR"].isnull() != punctuality["DELAY_ARR"].isnull()
print(f"Inconsistent rows after fix: {arr_rows_diff_after.sum()}")

Inconsistent rows after fix: 0


### 4.2.4.2. Handling Missing Values in Departure Columns

In [38]:
#Checking NaN for _DEP columns
print(
    (
        (punctuality["REAL_TIME_DEP"].isnull() == punctuality["PLANNED_TIME_DEP"].isnull()) &
        (punctuality["PLANNED_TIME_DEP"].isnull() == punctuality["REAL_DATE_DEP"].isnull()) &
        (punctuality["REAL_DATE_DEP"].isnull() == punctuality["PLANNED_DATE_DEP"].isnull()) &
        (punctuality["PLANNED_DATE_DEP"].isnull() == punctuality["DELAY_DEP"].isnull())
    ).all()
)

False


In [39]:
#Finding the problematical _DEP columns
ref_nan_dep = punctuality["REAL_TIME_DEP"].isnull()

cols_dep = ["PLANNED_TIME_DEP", "REAL_DATE_DEP", "PLANNED_DATE_DEP", "DELAY_DEP"]

for col in cols_dep:
    match = (punctuality[col].isnull() == ref_nan_dep).all()
    print(f"{col} : {match}")

PLANNED_TIME_DEP : False
REAL_DATE_DEP : True
PLANNED_DATE_DEP : False
DELAY_DEP : False


In [40]:
#Finding the problematical rows in DELAY_DEP
dep_rows_diff = punctuality["REAL_TIME_DEP"].isnull() != punctuality["DELAY_DEP"].isnull()
print(f"Inconsistent rows: {dep_rows_diff.sum()}")

dep_columns_to_check = (
 ["DATDEP", "RELATION", "PTCAR_LG_NM_NL", "REAL_TIME_DEP", "PLANNED_TIME_DEP", 
  "REAL_DATE_DEP", "PLANNED_DATE_DEP", "DELAY_DEP"]
)

punctuality.loc[dep_rows_diff, dep_columns_to_check].head()

Inconsistent rows: 105


,DATDEP,RELATION,PTCAR_LG_NM_NL,REAL_TIME_DEP,PLANNED_TIME_DEP,REAL_DATE_DEP,PLANNED_DATE_DEP,DELAY_DEP
648602,2024-01-11,ICE,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0
1448453,2024-01-24,ICE,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0
1747359,2024-01-28,IC 35,BRUSSEL-ZUID,<NA>,<NA>,NaT,NaT,0.0
2022813,2024-02-01,L L2,FLEMALLE-HAUTE,<NA>,<NA>,NaT,NaT,0.0
3295185,2024-02-21,L 26,LA LOUVIERE-SUD,<NA>,<NA>,NaT,NaT,0.0


The output above shows the same logical inconsistency found in `DELAY_ARR`: `0.0` values when `REAL_TIME_DEP` and `PLANNED_TIME_DEP` contain `<NA>` missing values.

In [41]:
#Finding the problematical rows in PLANNED_TIME_DEP and PLANNED_DATE_DEP 
dep_time_date_rows_diff = (
    (punctuality["REAL_TIME_DEP"].isnull() != punctuality["PLANNED_TIME_DEP"].isnull()) &
    (punctuality["REAL_DATE_DEP"].isnull() != punctuality["PLANNED_DATE_DEP"].isnull())
)
print(f"Inconsistent rows: {dep_time_date_rows_diff.sum()}")

punctuality.loc[dep_time_date_rows_diff, dep_columns_to_check].head()

Inconsistent rows: 1


,DATDEP,RELATION,PTCAR_LG_NM_NL,REAL_TIME_DEP,PLANNED_TIME_DEP,REAL_DATE_DEP,PLANNED_DATE_DEP,DELAY_DEP
19843209,2024-11-10,IC 24,LEUVEN,23:27:15,<NA>,2024-11-10,NaT,-71.0


As established at the beginning of this section, we drop this record from the dataset.

**⚠️ Warning**: the cell below takes approximately 4-5 minutes to execute on a standard machine.

In [42]:
punctuality = punctuality.drop(index=19843209).reset_index(drop=True)

In [43]:
punctuality.loc[dep_rows_diff, "DELAY_DEP"] = pd.NA

### 4.2.5. Checking for Orphan Station Keys between `punctuality` and `stations_cleaned`

As shown in the three sub-sections below, after cross-validating the `punctuality` and the `stations_cleaned` datasets, we find 5 **orphan values** in the future **Foreign Key** `PTCAR_NO`.  

These 5 values are also absent from the `operational_pts_railway` source dataset.  

Although these 5 orphan values represent a small fraction of the dataset, their presence in a future **Foreign Key** column requires thorough inquiry.  
  
| *Station Name*                | *Station ID*  | *Entries*     | *Share of `punctuality`* |
|-------------------------------|---------------|---------------|--------------------------|
| BAULERS                       | 125           | 11,225        | 0.025 %                  |
| MORTSEL-DEURNESTEENWEG        | 864           | 75,241        | 0.165 %                  |
| OLEN-SAS                      | 2222          | 32,879        | 0.072 %                  |
| ANTWERPEN-LINKEROEVER         | 2291          | 2362          | 0.005 %                  |
| BAASRODE-WIJKSPOREN           | 2300          | 1099          | 0.002 %                  |
| **Total number of entries**   | N/A           | **122,806**   | **0.27 %**               |
  

Upon investigation:  

* They correspond to **active stops**, accounting for 122,806 entries and 69 unique relation directions in `punctuality`.  

* The `operational_pts_railway` dataset was last updated on **February 21, 2025**, while `punctuality` covers the **full years 2024 and 2025**. Since orphan values appear before and after the last `operational_pts_railway` update, the update schedule cannot account for these orphan values.

An **in-depth analysis** of each orphan value is therefore conducted in the sub-section 4.2.5.4, at the end of this section.  


### 4.2.5.1. Cross-validating `punctuality` and `stations_cleaned` via `PTCAR_NO` to Detect Orphan Keys

In [44]:
stations = pd.read_parquet(PATH_INTERMEDIATE / "stations_cleaned.parquet")

In [45]:
punctuality_for_validation = punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]].drop_duplicates()
stations_for_validation = stations.drop(columns=["geo_point_2d", "geo_shape"])

In [46]:
except_merge = punctuality_for_validation.merge(
    stations_for_validation["ptcarid"],
    left_on="PTCAR_NO",
    right_on="ptcarid",
    how="left",
    indicator=True
)


In [47]:
orphan_stations_in_punctuality = (
    except_merge[except_merge["_merge"] == "left_only"].drop(columns=["_merge"]))

orphan_stations_in_punctuality

,PTCAR_NO,PTCAR_LG_NM_NL,ptcarid
112,864,MORTSEL-DEURNESTEENWEG,NaN
198,125,BAULERS,NaN
652,2222,OLEN-SAS,NaN
690,2291,ANTWERPEN-LINKEROEVER,NaN
691,2300,BAASRODE-WIJKSPOREN,NaN


As shown in the output above, there are 5 `PTCAR_NO` orphan values present in `punctuality` despite being absent from the `stations_cleaned` reference dataset.

In [48]:
id_orphan_stations_in_punctuality = orphan_stations_in_punctuality["PTCAR_NO"]

punctuality_orphans = punctuality[punctuality["PTCAR_NO"].isin(id_orphan_stations_in_punctuality)]

print(f"These orphan stations account for {punctuality_orphans.shape[0]} entries in the dataset.")

These orphan stations account for 122806 entries in the dataset.


Therefore, they were active in the railway network during the 2024-2025 time period.

### 4.2.5.2. Cross-validating `punctuality` with the Source Dataset via `PTCAR_NO`

We perform this cross-validation against the **raw data** to verify whether these **orphan keys** already exist in the source dataset from the *Infrabel Open Data Portal*.

In [49]:
stations_raw = pd.read_parquet(PATH_RAW / "operational_pts_railway_raw.parquet")
stations_raw["ptcarid"] = stations_raw["ptcarid"].astype("int64")

In [50]:
stations_raw_for_validation = stations_raw[["ptcarid", "class_en", "longnamefrench", "longnamedutch"]]

In [51]:
except_merge = punctuality_for_validation.merge(
    stations_raw_for_validation,
    left_on="PTCAR_NO",
    right_on="ptcarid",
    how="left",
    indicator=True
)


In [52]:
orphan_stations_in_punctuality_raw = (
    except_merge[except_merge["_merge"] == "left_only"].drop(columns=["_merge"]))

In [53]:
orphan_stations_in_punctuality_raw

,PTCAR_NO,PTCAR_LG_NM_NL,ptcarid,class_en,longnamefrench,longnamedutch
112,864,MORTSEL-DEURNESTEENWEG,NaN,NaN,NaN,NaN
198,125,BAULERS,NaN,NaN,NaN,NaN
652,2222,OLEN-SAS,NaN,NaN,NaN,NaN
690,2291,ANTWERPEN-LINKEROEVER,NaN,NaN,NaN,NaN
691,2300,BAASRODE-WIJKSPOREN,NaN,NaN,NaN,NaN


As shown above, these 5 orphan `PTCAR_NO` values are **likewise** absent from the source data.

### 4.2.5.3. Cross-validating on Station Names with the Source Data 

To verify whether these orphan keys are not present in the source data under different IDs, we cross-validate `punctuality` and the source dataset using **station names** instead of the `PTCAR_NO` values.

In [54]:
punctuality_for_validation["PTCAR_LG_NM_NL"] = (
    ip.clean_column_string(punctuality_for_validation["PTCAR_LG_NM_NL"])
    )

In [55]:
stations_raw_for_validation["longnamedutch"] = (
    ip.clean_column_string(stations_raw_for_validation["longnamedutch"])
)

In [56]:
except_merge = punctuality_for_validation.merge(
    stations_raw_for_validation,
    left_on="PTCAR_LG_NM_NL",
    right_on="longnamedutch",
    how="left",
    indicator=True
)

In [57]:
orphan_station_names_in_punctuality = (
    except_merge[except_merge["_merge"] == "left_only"].drop(columns=["_merge"])
)

orphan_station_names_in_punctuality

,PTCAR_NO,PTCAR_LG_NM_NL,ptcarid,class_en,longnamefrench,longnamedutch
112,864,mortsel-deurnesteenweg,NaN,NaN,NaN,NaN
198,125,baulers,NaN,NaN,NaN,NaN
258,196,haversin-garage,NaN,NaN,NaN,NaN
652,2222,olen-sas,NaN,NaN,NaN,NaN
679,339,montzen-n,NaN,NaN,NaN,NaN
690,2291,antwerpen-linkeroever,NaN,NaN,NaN,NaN
691,2300,baasrode-wijksporen,NaN,NaN,NaN,NaN


In the output above, there are 7 orphan station names instead of 5. Two of them actually have matching IDs in both datasets, and their mismatch is related to name standardization issues between the datasets: 
* *Haversin-garage* (ptcar_no **196**)
* *Montzen-n* (ptcar_no **339**)  

Since the **FK** between these datasets relies on `PTCAR_NO` in `punctuality` and `ptcarid` in `stations_cleaned`, these standardization issues have no impact and therefore do not need to be addressed - the displayed station names will use the station name columns from the *station dimension*, not from the *punctuality fact table*.


In [58]:
# Free up memory by deleting intermediate DataFrames no longer needed
del punctuality_for_validation
del stations_for_validation
del stations_raw_for_validation

### 4.2.5.4. In-depth Analysis of the Orphan `PTCAR_NO` Keys

To conduct this **in-depth analysis**:

* We identify the **infrastructure lines** where the orphan `PTCAR_NO` values are located and find their **nearest stations**. 

* We compare the **activity period** during which passenger trains was served these orphans stops and their nearest stations, in order to verify whether any of these orphans were temporary train stops replacing one of the nearest stations during engineering works.

The **table below** shows the **nearest stations** to the orphan `PTCAR_NO` values.  
  The *previous PTCAR* corresponds to the upstream station, from the nearest network hub (e.g. Brussels or Antwerp), while the *next PTCAR* corresponds to the downstream station relative to the same network hub.  

| Orphan PTCAR  | Orphan Station Name     | Prev. PTCAR | Prev. St. Name     | Next PTCAR | Next St. Name     |
|---------------|-------------------------|-------------|--------------------|------------|-------------------|
| 125           | BAULERS                 | 911         | NIVELLES           | N/A        | N/A               |
| 864           | MORTSEL-DEURNESTEENWEG  | 139         | ANTWERPEN-BERCHEM  | 866        | MORTSEL-OUDE GOD  |
| 2222          | OLEN-SAS                | 554         | HERENTALS          | 924        | OLEN              |
| 2291          | ANTWERPEN-LINKEROEVER   | 64          | ANTWERPEN-ZUID     | 1278       | ZWIJNDRECHT       |
| 2300          | BAASRODE-WIJKSPOREN     | 102         | BAASRODE-ZUID      | 319        | DENDERMONDE       |

<br><br>
The **table below** reveal the **activity periods** and the **number of train arrivals** for these stations:  
Bold **PTCAR_NO**, **Station Name**, and **Timeperiod** values identiy **orphan keys**.  

| PTCAR_NO  | Station Name               | Start Date   | End Date      | Timeperiod    | N° Train Arrivals | 
|-----------|----------------------------|--------------|---------------|---------------|-------------------|
| **125**   | **BAULERS**                | 2024-01-01   | 2024-08-30    | **242 days**  | 11,225            | 
| 911       | NIVELLES                   | 2024-01-01   | 2025-12-31    | 730 days      | 81,967            |
| **864**   | **MORTSEL-DEURNESTEENWEG** | 2024-01-01   | 2024-12-14    | **348 days**  | 75,241            |
| 139       | ANTWERPEN-BERCHEM          | 2024-01-01   | 2025-12-31    | 730 days      | 459,792           |
| 866       | MORTSEL-OUDE GOD           | 2024-01-01   | 2025-12-31    | 730 days      | 154,890           |
| 863       | MORTSEL                    | 2024-01-01   | 2025-12-31    | 730 days      | 248,865           |
| **2222**  | **OLEN-SAS**               | 2024-06-09   | 2025-12-31    | **570 days**  | 32,879            |  
| 554       | HERENTALS                  | 2024-01-01   | 2025-12-31    | 730 days      | 81,637            |
| 924       | OLEN                       | 2024-01-01   | 2025-12-31    | 730 days      | 41,491            |
| **2291**  | **ANTWERPEN-LINKEROEVER**  | 2025-12-14   | 2025-12-31    | **17 days**   | 2362              |
| 64        | ANTWERPEN-ZUID             | 2024-01-01   | 2025-12-31    | 730 days      | 96,763            |
| 1278      | ZWIJNDRECHT                | 2024-01-01   | 2025-12-31    | 730 days      | 96,765            |
| **2300**  | **BAASRODE-WIJKSPOREN**    | 2025-12-14   | 2025-12-31    | **17 days**   | 1099              |
| 102       | BAASRODE-ZUID              | 2024-01-01   | 2025-12-31    | 730 days      | 47,205            |
| 319       | DENDERMONDE                | 2024-01-01   | 2025-12-31    | 730 days      | 46,847            |

<br>

**A. Orphan Key `125` `BAULERS`**

* **BAULERS** (`PTCAR_NO` `125`) was a historical station, near **NIVELLES** (`PTCAR_NO` `911`), which became a stop after 1984 and was permanently closed in 1993. However, it was once again active between January 1, 2024, and August 30, 2024, serving as a **railway siding** on `124L/3` and `124L/4` infrastructure lines. 

* During the same period, **25,631** trains arrived at **NIVELLES** station. Therefore, we can discard the hypothesis that **BAULERS** was a subsitute for **NIVELLES** during engineering works.

**B. Orphan Key `864` `MORTSEL-DEURNESTEENWEG`**

* According to *SNCB documentation*, **MORTSEL_DEUNERSTEENWEG** merged with **MORTSEL** (`ptcarid` **863**) on December 15, 2024. Nevertheless, since **MORTSEL** station is served by the same train line, adding the **MORTSEL_DEUNERSTEENWEG** entries to it would introduce a **statistical bias**, as explained in the conclusion below.

**C. Orphan Key `2222` `OLEN-SAS`**

* `OLEN-SAS` appears to designate a halt located within a signal overlap section adjacent to `OLEN` station, as indicated by the suffix *"-SAS"*. This suffix also appears in three other entries in the `stations_cleaned` dataset: `RHISNES-SAS` (`PTCAR_NO` **430**), `SCHULEN-SAS` (`PTCAR_NO` **465**), and `IZEGEM-SAS` (`PTCAR_NO` **672**).

* `OLEN-SAS` was active from June 9, 2024, to December 31, 2025. During the same period, the exact same number of trains stopped both at `OLEN-SAS` and at the downstream `OLEN` station - **33,879**, while **64,507** trains stopped at the upstream `HERENTALS` station. As with `MORTSEL-DEURNESTEENWEG`, adding the  `OLEN-SAS` **entries** to either nearest neighbor would introduce a **statistical bias**.

**D. Orphan Key `2291` `ANTWERPEN-LINKEROEVER`**

* According to *SNCB documentation*, **ANTWERPEN-LINKEROEVER** was a historical `stop in open track` west of Antwerp, on the ANTWERPEN-CENTRAAL line, located between **ANTWERPEN-ZUID** (`PTCAR_NO` **64**) and **ZWIJNDRECHT** stations (`PTCAR_NO` **1278**). Closed in 1984, it briefly reappeared between 14 December and 31 December, 2025, then was closed again. It is reopen once more to passengers since 4th may, 2026, which is outside the scope of our analysis.

**E. Orphan Key `2300` `BAASRODE-WIJKSPOREN`**

* **BAASRODE-WIJKSPOREN** appears to be a **railway siding** near **BAASRODE-ZUID** station (`PTCAR_NO` **102**).

* This stop was active at the end of 2025, precisely from 14 December to 31 December, 2025. As 2026 falls out of the scope of this analysis, its status in the following months remains unclear. It may have been opened temporarily during the end-of-year holiday period, given its location on the same line serving the university town of Leuven, though no evidence supports this hypothesis.
  
<br>

**CONCLUSION**:

* There is a **discrepancy** between the `punctuality` dataset and the **master data** that references all Infrabel operational points - including technical points - in the `operational_pts_railway` source dataset.  

* All orphan keys were **active** at some point between 2024 and 2025. Their entries correspond to **real passenger stops**. Consequently, they cannot be excluded from the analysis.  

* In no case is an orphan key active during a period when any of its nearest stations is inactive. These orphan keys are therefore not temporary replacement stops for stations closed during engineering works.

* Since some of these orphan keys are railway sidings that may be used in case of incidents or delay, such as **BAULERS** and **BAASRODE-WIJKSPOREN**, and others were still active passenger stops during 2024 or 2025, we cannot assign them the `PTCAR_NO` value of their nearest station. If a train stops at one of them due to a delay, it will also be recorded as late at the next station on the same relation in `punctuality`. As a result, this would **duplicate the delay**, counting both the delay at the orphan key and the delay at its nearest station, inflating average delay statistics by station, by relation, or by geographical entity.  

**DECISION**:  
* In the *03_01_building_dimension_station* notebook, the following entries will be added:  

    * `125` `BAULERS`: `ptcarid` == `125`, `class_en` == `"Service installation"`, French and Dutch station name columns == `BAULERS`.  
    
    * `864` `MORTSEL-DEURNESTEENWEG`: `ptcarid` == `864`, `class_en` == `"Stop in open track"`, French and Dutch station name columns == `MORTSEL (former MORTSEL-DEURNESTEENWEG)`.

    * `2222` `OLEN-SAS`: `ptcarid` == `2222`, `class_en` == `"Station"`, French and Dutch station name columns == `"OLEN-SAS"`.

    * `2291` `ANTWERPEN-LINKEROEVER`: `ptcarid` == `2291`, `class_en` == `"Stop in open track"`, French and Dutch station name columns == `"ANTWERPEN-LINKEROEVER"`.

    * `2300` `BAASRODE-WIJKSPOREN`: `ptcarid` == `2300`, `class_en` == `"Unknown"`, French and Dutch station name columns == `"BAASRODE-WIJKSPOREN"`.

* For all five entries, geographical coordinates will be set to `<NA>` in Python and `NULL` in the SQL data warehouse, as these coordinates are absent from the source data.

* A binary **`Is_Inferred`** column will be added to allow excluding these five inferred keys by filtering in Power BI.

* The definitive handling of these orphan keys will therefore require **further enrichment** in the *03_01_build_dimension_station* notebook.

In [59]:
orphan_stations_list = orphan_stations_in_punctuality_raw["PTCAR_NO"].drop_duplicates().sort_values().to_list()
orphan_stations_list

[125, 864, 2222, 2291, 2300]

In [60]:
number_entries_groupby_orphans = (
    punctuality_orphans.groupby("PTCAR_NO")["DATDEP"].agg(["count"])
)

number_entries_groupby_orphans["proportion_dataset"] = (
    (number_entries_groupby_orphans["count"] / punctuality.shape[0] * 100).round(3)
    )

display(number_entries_groupby_orphans)

print(f"The total number of entries for these orphan key values is {number_entries_groupby_orphans["count"].sum()}.")

print(f"The total proportion of entries for these orphan key values is "
      f"{(number_entries_groupby_orphans["count"].sum() / punctuality.shape[0] * 100).round(3)} % of the dataset.")

,count,proportion_dataset
PTCAR_NO,,
125,11225,0.025
864,75241,0.165
2222,32879,0.072
2291,2362,0.005
2300,1099,0.002


The total number of entries for these orphan key values is 122806.
The total proportion of entries for these orphan key values is 0.27 % of the dataset.


**A. `125` `BAULERS` Orphan Key Analysis**

In [61]:
punctuality_125 = punctuality.loc[punctuality["PTCAR_NO"] == 125]

In [62]:
punctuality_125[["LINE_NO_ARR", "LINE_NO_DEP"]].dropna().drop_duplicates().reset_index(drop=True)

,LINE_NO_ARR,LINE_NO_DEP
0,124L/3,124L/3
1,124L/4,124L/4


`125` `BAULERS` is served by the **124L/3** and **124L/4** train lines.

In [63]:
ptcar_124L_lines = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]]
    .loc[punctuality["LINE_NO_ARR"].isin(["124L/3", "124L/4"])]
    .drop_duplicates().reset_index(drop=True)
)
ptcar_124L_lines

,PTCAR_NO,PTCAR_LG_NM_NL
0,125,BAULERS
1,911,NIVELLES


    Only two `PTCAR_NO` unique values are served by these train lines:  
    * **125** for *Baulers*  
    * **911** for *Nivelles*  

In [64]:
punctuality_124L_lines = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "DATDEP", "REAL_TIME_ARR"]]
    .loc[(punctuality["LINE_NO_ARR"].isin(["124L/3", "124L/4"])).fillna(False)]
    .drop_duplicates().reset_index(drop=True)
)

pctcar_ids_on_124L_lines_list = punctuality_124L_lines["PTCAR_NO"].to_list()

ptcar_125_and_911_timeperiod = (
    punctuality.loc[punctuality["PTCAR_NO"].isin(pctcar_ids_on_124L_lines_list)]
    .groupby("PTCAR_NO").agg(
        station_name=("PTCAR_LG_NM_NL", "first"),
        min=("DATDEP", "min"),
        max=("DATDEP", "max"),
        count=("REAL_TIME_ARR", "count")
        )
)

ptcar_125_and_911_timeperiod["timeperiod"] = (
    (ptcar_125_and_911_timeperiod["max"] - ptcar_125_and_911_timeperiod["min"])
)

ptcar_125_and_911_timeperiod

,station_name,min,max,count,timeperiod
PTCAR_NO,,,,,
125,BAULERS,2024-01-01,2024-08-30,11225,242 days
911,NIVELLES,2024-01-01,2025-12-31,81967,730 days


**BAULERS** and **NIVELLES** stations were both active between 1 January 2024 and 30 August 2024, but **BAULERS** ceased to be active after this date.

In [65]:
date_filter_911 = (punctuality["DATDEP"] >= "2024-01-01") & (punctuality["DATDEP"] <= "2024-08-30")

nivelles_active_count = (
    punctuality.loc[(punctuality["PTCAR_NO"] == 911) & (date_filter_911), "REAL_TIME_ARR"]
    .count()
)

print(f"Number of train arrivals at NIVELLES (911) from 2024-01-01 to 2024-08-30 : {nivelles_active_count}")

Number of train arrivals at NIVELLES (911) from 2024-01-01 to 2024-08-30 : 25631


The output above proves that NIVELLES station was likewise active during the same timeperiod.

In [66]:
# Free up memory by deleting intermediate DataFrame
del punctuality_125

**B. `864` `MORTSEL-DEURNESTEENWEG` Orphan Key Analysis**

In [67]:
punctuality_864 = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "LINE_NO_DEP", "PLANNED_TIME_ARR"]]
    .loc[punctuality["PTCAR_NO"] == 864]
)

In [68]:
print(f"Orphan value 864 in PTCAR_NO has {punctuality_864.shape[0]} entries in punctuality.")

Orphan value 864 in PTCAR_NO has 75241 entries in punctuality.


In [69]:
punctuality_864[["LINE_NO_ARR", "LINE_NO_DEP"]].dropna().drop_duplicates().reset_index(drop=True)

,LINE_NO_ARR,LINE_NO_DEP
0,25,25


`864` `MORTSEL-DEURNESTEENWEG` is only served by the **25** train line.

In [70]:
ptar_ids_on_25_line = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]]
    .loc[punctuality["LINE_NO_ARR"] == "25"]
    .drop_duplicates().reset_index(drop=True)
)
print(f"Line 25 accounts for {len(ptar_ids_on_25_line["PTCAR_NO"])} stations or train stops.")

Line 25 accounts for 18 stations or train stops.


The code below calls `finding_nearest_stations()` on **line 25** and returns the upstream and downstream stations of orphan key **864**.

In [71]:
punctuality_line25 = (
    punctuality
    .loc[
        (punctuality["LINE_NO_ARR"] == "25").fillna(False),
        ["DATDEP", "TRAIN_NO", "PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "PLANNED_TIME_ARR"]
    ]
)

In [72]:
prev_counts_864, next_counts_864 = ip.finding_nearest_stations(punctuality_line25, 864)

=== Stations preceding 864 ===
 PREV_PTCAR_NO        PREV_PTCAR_NM  count
           866     MORTSEL-OUDE GOD  37706
           139    ANTWERPEN-BERCHEM  36086
            37   ANTWERPEN-CENTRAAL     79
           221        BRUSSEL-NOORD     35
           811 MECHELEN-NEKKERSPOEL      3
          1048           SCHAARBEEK      2

=== Stations following 864 ===
 NEXT_PTCAR_NO        NEXT_PTCAR_NM  count
           866     MORTSEL-OUDE GOD  37531
           139    ANTWERPEN-BERCHEM  37180
            37   ANTWERPEN-CENTRAAL     80
          1048           SCHAARBEEK     20
           229                 BUDA      6
           590                 HOVE      1
           810             MECHELEN      1
          1083 SINT-KATELIJNE-WAVER      1


* `864` `MORTSEL-DEURNESTEENWEG` is located between `139` `ANTWERPEN-BERCHEM`, upstream, and `866` `MORTSEL-OUDE GOD`, downstream from the Antwerp network hub.   

* The other stations are statistically irrelevant, probably `EXTRA`, `CHARTER` or international train services.

In [73]:
nearest_stations_864_list = [864, 866, 139, 863]

ptcar_864_and_neighbors_timeperiod = (
    punctuality.loc[punctuality["PTCAR_NO"].isin(nearest_stations_864_list)]
    .groupby("PTCAR_NO").agg(
        station_name=("PTCAR_LG_NM_NL", "first"),
        min=("DATDEP", "min"),
        max=("DATDEP", "max"),
        count=("REAL_TIME_ARR", "count")
        )
)

ptcar_864_and_neighbors_timeperiod["timeperiod"] = (
    (ptcar_864_and_neighbors_timeperiod["max"] - ptcar_864_and_neighbors_timeperiod["min"])
)

ptcar_864_and_neighbors_timeperiod

,station_name,min,max,count,timeperiod
PTCAR_NO,,,,,
139,ANTWERPEN-BERCHEM,2024-01-01,2025-12-31,459792,730 days
863,MORTSEL,2024-01-01,2025-12-31,248865,730 days
864,MORTSEL-DEURNESTEENWEG,2024-01-01,2024-12-14,75241,348 days
866,MORTSEL-OUDE GOD,2024-01-01,2025-12-31,154890,730 days


In [74]:
date_filter_864 = (punctuality["DATDEP"] >= "2024-01-01") & (punctuality["DATDEP"] <= "2024-12-14")

active_count_139 = (
    punctuality.loc[(punctuality["PTCAR_NO"] == 139) & (date_filter_864), "REAL_TIME_ARR"]
    .count()
)
active_count_866 = (
    punctuality.loc[(punctuality["PTCAR_NO"] == 866) & (date_filter_864), "REAL_TIME_ARR"]
    .count()
)
active_count_863 = (
    punctuality.loc[(punctuality["PTCAR_NO"] == 863) & (date_filter_864), "REAL_TIME_ARR"]
    .count()
)
active_count_864 = (
    punctuality.loc[(punctuality["PTCAR_NO"] == 864) & (date_filter_864), "REAL_TIME_ARR"]
    .count()
)

print(f"Number of train arrivals at ANTWERPEN-BERCHEM (139) from 2024-01-01 to 2024-12-14: {active_count_139}")
print(f"Number of train arrivals at MORTSEL-OUDE GOD (866) from 2024-01-01 to 2024-12-14: {active_count_866}")
print(f"Number of train arrivals at MORTSEL (863) from 2024-01-01 to 2024-12-14: {active_count_863}")
print(f"Number of train arrivals at the orphan key MORTSEL-DEURNESTEENWEG (864) from 2024-01-01 to 2024-12-14: {active_count_864}")

Number of train arrivals at ANTWERPEN-BERCHEM (139) from 2024-01-01 to 2024-12-14: 223318
Number of train arrivals at MORTSEL-OUDE GOD (866) from 2024-01-01 to 2024-12-14: 75240
Number of train arrivals at MORTSEL (863) from 2024-01-01 to 2024-12-14: 80876
Number of train arrivals at the orphan key MORTSEL-DEURNESTEENWEG (864) from 2024-01-01 to 2024-12-14: 75241


In [75]:
# Free up memory by deleting intermediate DataFrame
del punctuality_864

**C. `2222` `OLEN-SAS` Orphan Key Analysis**

In [76]:
punctuality_2222 = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "LINE_NO_DEP", "PLANNED_TIME_ARR"]]
    .loc[punctuality["PTCAR_NO"] == 2222]
)

In [77]:
print(f"Orphan value 2222 in PTCAR_NO accounts for {punctuality_2222.shape[0]} entries in punctuality.")

Orphan value 2222 in PTCAR_NO accounts for 32879 entries in punctuality.


In [78]:
punctuality_2222[["LINE_NO_ARR", "LINE_NO_DEP"]].dropna().drop_duplicates().reset_index(drop=True)

,LINE_NO_ARR,LINE_NO_DEP
0,15,15


`2222` `OLEN-SAS` is served by the **15** train line.

In [79]:
ptcar_2222_line = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]]
    .loc[punctuality["LINE_NO_ARR"] == "15"]
    .drop_duplicates().reset_index(drop=True)
)
print(f"The line 15 accounts for {len(ptcar_2222_line["PTCAR_NO"])} stations or train stops.")

The line 15 accounts for 19 stations or train stops.


The code below calls `finding_nearest_stations()` on **line 15** and returns the upstream and  downstream stations of orphan key **2222**.

In [80]:
punctuality_line15 = (
    punctuality
    .loc[
        (punctuality["LINE_NO_ARR"] == "15").fillna(False),
        ["DATDEP", "TRAIN_NO", "PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "PLANNED_TIME_ARR", "REAL_TIME_ARR"]
    ]
)

In [81]:
prev_counts_2222, next_counts_2222 = ip.finding_nearest_stations(punctuality_line15, 2222)

=== Stations preceding 2222 ===
 PREV_PTCAR_NO PREV_PTCAR_NM  count
           924          OLEN  16519
           554     HERENTALS  16091
           840           MOL     29

=== Stations following 2222 ===
 NEXT_PTCAR_NO NEXT_PTCAR_NM  count
           554     HERENTALS  16519
           924          OLEN  16360


* `2222` `OLEN-SAS` is located between `554` `HERENTALS`, upstream, and `924` `OLEN`, downstream from the Antwerp network hub. 

In [82]:
nearest_stations_2222_list = [2222, 554, 924]

ptcar_2222_and_neighbors_timeperiod = (
    punctuality_line15.loc[punctuality["PTCAR_NO"].isin(nearest_stations_2222_list)]
    .groupby("PTCAR_NO").agg(
        station_name=("PTCAR_LG_NM_NL", "first"),
        min=("DATDEP", "min"),
        max=("DATDEP", "max"),
        count=("REAL_TIME_ARR", "count")
        )
)

ptcar_2222_and_neighbors_timeperiod["timeperiod"] = (
    (ptcar_2222_and_neighbors_timeperiod["max"] - ptcar_2222_and_neighbors_timeperiod["min"])
)

ptcar_2222_and_neighbors_timeperiod

,station_name,min,max,count,timeperiod
PTCAR_NO,,,,,
554,HERENTALS,2024-01-01,2025-12-31,81637,730 days
924,OLEN,2024-01-01,2025-12-31,41491,730 days
2222,OLEN-SAS,2024-06-09,2025-12-31,32879,570 days


In [83]:
date_filter_2222 = (punctuality_line15["DATDEP"] >= "2024-06-09") & (punctuality_line15["DATDEP"] <= "2025-12-31")

active_count_924 = (
    punctuality_line15.loc[(punctuality_line15["PTCAR_NO"] == 924) & (date_filter_2222), "REAL_TIME_ARR"]
    .count()
)
active_count_554 = (
    punctuality_line15.loc[(punctuality_line15["PTCAR_NO"] == 554) & (date_filter_2222), "REAL_TIME_ARR"]
    .count()
)
active_count_2222 = (
    punctuality_line15.loc[(punctuality_line15["PTCAR_NO"] == 2222) & (date_filter_2222), "REAL_TIME_ARR"]
    .count()
)

print(f"Number of train arrivals at OLEN (924) from 2024-06-09 to 2025-12-31: {active_count_924}")
print(f"Number of train arrivals at HERENTALS (554) from 2024-06-09 to 2025-12-31: {active_count_554}")
print(f"Number of train arrivals at OLEN-SAS (2222) from 2024-06-09 to 2025-12-31: {active_count_2222}")

Number of train arrivals at OLEN (924) from 2024-06-09 to 2025-12-31: 32879
Number of train arrivals at HERENTALS (554) from 2024-06-09 to 2025-12-31: 64507
Number of train arrivals at OLEN-SAS (2222) from 2024-06-09 to 2025-12-31: 32879


Exactly the same number of trains arrived at **OLEN** and **OLEN-SAS** stations during the same time period.

In [84]:

idx_sas_in_station_fr = stations["longnamefrench"].str.extract(r'(SAS)$', expand=False).dropna()
idx_sas_in_station_nl = stations["longnamedutch"].str.extract(r'(SAS)$', expand=False).dropna()
check_fr_nl_variance = (stations.loc[idx_sas_in_station_fr.index, :]) == (stations.loc[idx_sas_in_station_nl.index, :])
display(check_fr_nl_variance.all())
sas_in_station = stations.loc[idx_sas_in_station_fr.index, :]
sas_in_station[["longnamefrench", "class_en"]]


geo_point_2d      True
geo_shape         True
ptcarid           True
longnamefrench    True
longnamedutch     True
class_en          True
dtype: bool

,longnamefrench,class_en
430,RHISNES-SAS,Station
465,SCHULEN-SAS,Station
672,IZEGEM-SAS,Station


In [85]:
# Free up memory by deleting intermediate DataFrame
del punctuality_2222

**D. `2291` `ANTWERPEN-LINKEROEVER` Orphan Key Analysis**

In [86]:
punctuality_2291 = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "LINE_NO_DEP", "PLANNED_TIME_ARR"]]
    .loc[punctuality["PTCAR_NO"] == 2291]
)

In [87]:
print(f"Orphan value 2291 in PTCAR_NO accounts for {punctuality_2291.shape[0]} entries in punctuality.")

Orphan value 2291 in PTCAR_NO accounts for 2362 entries in punctuality.


In [88]:
punctuality_2291[["LINE_NO_ARR", "LINE_NO_DEP"]].dropna().drop_duplicates().reset_index(drop=True)

,LINE_NO_ARR,LINE_NO_DEP
0,59,59


`2291` `ANTWERPEN-LINKEROEVER` is served by the **59** train line.

In [89]:
ptcar_59_line = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]]
    .loc[punctuality["LINE_NO_ARR"] == "59"]
    .drop_duplicates().reset_index(drop=True)
)
print(f"The line 15 accounts for {len(ptcar_59_line["PTCAR_NO"])} stations or train stops.")

The line 15 accounts for 15 stations or train stops.


The code below calls `finding_nearest_stations()` on **line 15** and returns the upstream and downstream stations of orphan key **2291**.

In [90]:
punctuality_line59 = (
    punctuality
    .loc[
        (punctuality["LINE_NO_ARR"] == "59").fillna(False),
        ["DATDEP", "TRAIN_NO", "PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "PLANNED_TIME_ARR", "REAL_TIME_ARR"]
    ]
)

In [91]:
prev_counts_2291, next_counts_2291 = ip.finding_nearest_stations(punctuality_line59, 2291)

=== Stations preceding 2291 ===
 PREV_PTCAR_NO  PREV_PTCAR_NM  count
            64 ANTWERPEN-ZUID   1182
          1278    ZWIJNDRECHT   1180

=== Stations following 2291 ===
 NEXT_PTCAR_NO  NEXT_PTCAR_NM  count
          1278    ZWIJNDRECHT   1182
            64 ANTWERPEN-ZUID   1180


* `2291` `ANTWERPEN-LINKEROEVER` is located between `64` `ANTWERPEN-ZUID`, upstream, and `1278` `ZWIJNDRECHT`, downstream. 

In [92]:
nearest_stations_2291_list = [2291, 64, 1278]

ptcar_2291_and_neighbors_timeperiod = (
    punctuality_line59.loc[punctuality["PTCAR_NO"].isin(nearest_stations_2291_list)]
    .groupby("PTCAR_NO").agg(
        station_name=("PTCAR_LG_NM_NL", "first"),
        min=("DATDEP", "min"),
        max=("DATDEP", "max"),
        count=("REAL_TIME_ARR", "count")
        )
)

ptcar_2291_and_neighbors_timeperiod["timeperiod"] = (
    (ptcar_2291_and_neighbors_timeperiod["max"] - ptcar_2291_and_neighbors_timeperiod["min"])
)

ptcar_2291_and_neighbors_timeperiod

,station_name,min,max,count,timeperiod
PTCAR_NO,,,,,
64,ANTWERPEN-ZUID,2024-01-01,2025-12-31,96763,730 days
1278,ZWIJNDRECHT,2024-01-01,2025-12-31,96765,730 days
2291,ANTWERPEN-LINKEROEVER,2025-12-14,2025-12-31,2362,17 days


* Train arrivals in `ANTWERPEN-LINKEROEVER` correspond to **a small proportion of the train arrivals of its nearest stations**.

* Since this station was open only **for 17 days** at the end of 2025, its train arrival count is not directly comparable to those of its neighbors.

In [93]:
# Free up memory by deleting intermediate DataFrame
del punctuality_2291

**E. `2300` `BAASRODE-WIJKSPOREN` Orphan Key Analysis**

In [94]:
punctuality_2300 = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "LINE_NO_DEP", "PLANNED_TIME_ARR"]]
    .loc[punctuality["PTCAR_NO"] == 2300]
)

In [95]:
print(f"Orphan value 2300 in PTCAR_NO accounts for {punctuality_2300.shape[0]} entries in punctuality.")

Orphan value 2300 in PTCAR_NO accounts for 1099 entries in punctuality.


In [96]:
punctuality_2300[["LINE_NO_ARR", "LINE_NO_DEP"]].dropna().drop_duplicates().reset_index(drop=True)

,LINE_NO_ARR,LINE_NO_DEP
0,53,53


`2300` `BAASRODE-WIJKSPOREN` is served by the **53** train line.

In [97]:
ptcar_2300_line = (
    punctuality[["PTCAR_NO", "PTCAR_LG_NM_NL"]]
    .loc[punctuality["LINE_NO_ARR"] == "53"]
    .drop_duplicates().reset_index(drop=True)
)
print(f"The line 53 accounts for {len(ptcar_2300_line["PTCAR_NO"])} stations or train stops.")

The line 53 accounts for 21 stations or train stops.


The code below calls `finding_nearest_stations()` on **line 53** and returns the upstream and downstream stations of orphan key **2300**.

In [98]:
punctuality_line53 = (
    punctuality
    .loc[
        (punctuality["LINE_NO_ARR"] == "53").fillna(False),
        ["DATDEP", "TRAIN_NO", "PTCAR_NO", "PTCAR_LG_NM_NL", "LINE_NO_ARR", "PLANNED_TIME_ARR", "REAL_TIME_ARR"]
    ]
)

In [99]:
prev_counts_2300, next_counts_2300 = ip.finding_nearest_stations(punctuality_line53, 2300)

=== Stations preceding 2300 ===
 PREV_PTCAR_NO PREV_PTCAR_NM  count
           102 BAASRODE-ZUID    547
           319   DENDERMONDE    498

=== Stations following 2300 ===
 NEXT_PTCAR_NO NEXT_PTCAR_NM  count
           102 BAASRODE-ZUID    552
           319   DENDERMONDE    517


* `2300` `BAASRODE-WIJKSPOREN` is located between `102` `BAASRODE-ZUID`, upstream, and `319` `DENDERMONDE`, downstream, from the Leuven University Town.

In [100]:
nearest_stations_2300_list = [2300, 102, 319]

ptcar_2300_and_neighbors_timeperiod = (
    punctuality_line53.loc[punctuality["PTCAR_NO"].isin(nearest_stations_2300_list)]
    .groupby("PTCAR_NO").agg(
        station_name=("PTCAR_LG_NM_NL", "first"),
        min=("DATDEP", "min"),
        max=("DATDEP", "max"),
        count=("REAL_TIME_ARR", "count")
        )
)

ptcar_2300_and_neighbors_timeperiod["timeperiod"] = (
    (ptcar_2300_and_neighbors_timeperiod["max"] - ptcar_2300_and_neighbors_timeperiod["min"])
)

ptcar_2300_and_neighbors_timeperiod

,station_name,min,max,count,timeperiod
PTCAR_NO,,,,,
102,BAASRODE-ZUID,2024-01-01,2025-12-31,47205,730 days
319,DENDERMONDE,2024-01-01,2025-12-31,46847,730 days
2300,BAASRODE-WIJKSPOREN,2025-12-14,2025-12-31,1099,17 days


* Train arrivals in `BAASRODE-WIJKSPOREN` correspond to **a small proportion of the train arrivals of its nearest stations**.

* Since this station was open only **for 17 days** at the end of 2025, its train arrival count is not directly comparable to those of its neighbors.

In [101]:
# Free up memory by deleting intermediate DataFrame
del punctuality_2300

## 4.3. Export to Silver Layer

In [102]:
punctuality.to_parquet(PATH_INTERMEDIATE / "punctuality_checkpoint_02_04.parquet")

In [103]:
# Free up memory by deleting intermediate DataFrame
del punctuality

In [104]:
df_to_verify = pd.read_parquet(PATH_INTERMEDIATE / "punctuality_checkpoint_02_04.parquet")

In [105]:
print(df_to_verify.shape, df_to_verify.dtypes.to_dict())
df_to_verify.head()

(45542084, 19) {'DATDEP': dtype('<M8[us]'), 'TRAIN_NO': Int64Dtype(), 'RELATION': <StringDtype(na_value=<NA>)>, 'TRAIN_SERV': <StringDtype(na_value=<NA>)>, 'PTCAR_NO': Int64Dtype(), 'LINE_NO_DEP': <StringDtype(na_value=<NA>)>, 'REAL_TIME_ARR': <StringDtype(na_value=<NA>)>, 'REAL_TIME_DEP': <StringDtype(na_value=<NA>)>, 'PLANNED_TIME_ARR': <StringDtype(na_value=<NA>)>, 'PLANNED_TIME_DEP': <StringDtype(na_value=<NA>)>, 'DELAY_ARR': dtype('float64'), 'DELAY_DEP': dtype('float64'), 'RELATION_DIRECTION': <StringDtype(na_value=<NA>)>, 'PTCAR_LG_NM_NL': <StringDtype(na_value=<NA>)>, 'LINE_NO_ARR': <StringDtype(na_value=<NA>)>, 'PLANNED_DATE_ARR': dtype('<M8[us]'), 'PLANNED_DATE_DEP': dtype('<M8[us]'), 'REAL_DATE_ARR': dtype('<M8[us]'), 'REAL_DATE_DEP': dtype('<M8[us]')}


,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP
0,2024-01-01,1813,IC 02,SNCB/NMBS,929,50A,<NA>,13:09:42,<NA>,13:09:00,NaN,42.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,<NA>,NaT,2024-01-01,NaT,2024-01-01
1,2024-01-01,1813,IC 02,SNCB/NMBS,210,50A,13:22:17,13:25:20,13:22:00,13:25:00,17.0,20.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
2,2024-01-01,1813,IC 02,SNCB/NMBS,931,50A,13:29:18,13:29:18,13:28:00,13:28:00,78.0,78.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
3,2024-01-01,1813,IC 02,SNCB/NMBS,127,50A,13:31:59,13:31:59,13:31:00,13:31:00,59.0,59.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
4,2024-01-01,1813,IC 02,SNCB/NMBS,797,50A,13:33:57,13:33:57,13:33:00,13:33:00,57.0,57.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01
